## Genre Cleaning and Standardization

This section refines and standardizes genre data to improve consistency and usability for downstream tasks.

---

### Data Loading

- Loads the dataset containing tracks and their associated genre tags  
- Prepares the data for cleaning and transformation  

---

### Genre Normalization

- Defines mappings between different variations of the same genre  
- Converts multiple representations into a single canonical form  
- Ensures consistency across similar genre labels  

---

### Variant Mapping

- Builds a lookup structure to map all known genre variants to their standardized form  
- Handles differences in spelling, spacing, and formatting  

---

### Spacing Normalization

- Fixes inconsistencies caused by missing or incorrect spacing  
- Converts compact forms into readable genre labels  

---

### Genre Filtering

- Removes invalid or noisy genre entries based on:
  - Presence of numeric characters  
  - Excessively long genre phrases  

---

### Genre Cleaning

- Splits genre strings into individual labels  
- Applies normalization and mapping  
- Filters invalid entries  
- Removes duplicates and sorts results  

---

### Artist Name Removal

- Detects and removes genre tags that match or closely resemble the artist name  
- Uses token-based comparison to identify overlaps  
- Ensures only meaningful genre information is retained  

---

### Final Output

- Stores cleaned and standardized genres in a new column  
- Saves the fully processed dataset to a new file  

---

### Purpose

- Improves data quality and consistency  
- Reduces noise and redundancy in genre labels  
- Prepares structured input for recommendation models and analysis  

In [46]:
import pandas as pd
import re

# ---------- LOAD ----------
df = pd.read_csv("songs_with_genres.csv")

# ---------- GENRE EQUIVALENTS ----------
GENRE_EQUIVALENTS = {
    'pop punk': ['pop punk', 'pop-punk', 'punk pop'],
    'west coast hip hop': ['west coast rap', 'west coast hip hop', 'west coast hip-hop'],
    'pop rock': ['pop rock', 'pop-rock', 'rock pop'],
    'nu metal': ['nu metal', 'nu-metal'],
    'post grunge': ['post-grunge', 'post grunge'],
    'dance pop': ['dance-pop', 'dance pop'],
    'underground hip hop': ['underground hip-hop', 'underground hip hop', 'underground rap'],
    'jazz hop': ['jazz hop', 'jazz-hop'],
    'singer songwriter': ['singer-songwriter', 'singer songwriter'],
    'east coast hip hop': ['east coast rap', 'east coast hip hop', 'east coast hip-hop'],
    'neo soul': ['neo-soul', 'neo soul'],
    'alt country': ['alt-country', 'alt country'],
    'lo fi': ['lo-fi', 'lo fi'],
    'vocal jazz': ['vocal jazz', 'jazz vocal'],
    'alternative hip hop': ['alternative rap', 'alternative hip hop', 'alternative hip-hop', 'alternative hiphop'],
    'southern hip hop': ['southern hip hop', 'southern hip-hop', 'southern rap'],
    'hardcore hip hop': ['hardcore hip hop', 'hardcore rap', 'hardcore hip-hop', 'hardcore hiphop'],
    'conscious hip hop': ['conscious hip hop', 'conscious rap', 'conscious hip-hop'],
    'experimental hip hop': ['experimental hip hop', 'experimental hip-hop', 'experimental rap'],
    'trip hop': ['trip-hop', 'trip hop'],
    'post hardcore': ['post-hardcore', 'post hardcore'],
    'synth pop': ['synth pop', 'synth-pop'],
    'post punk': ['post-punk', 'post punk'],
    'folk rock': ['folk rock', 'folk-rock'],
    'latin hip hop': ['latin rap', 'latin hip hop'],
    'doo wop': ['doo wop', 'doo-wop'],
    'country pop': ['country pop', 'country-pop', 'pop country'],
    'pop rap': ['pop rap', 'pop-rap'],
    'blue eyed soul': ['blue-eyed soul', 'blue eyed soul'],
    'abstract hip hop': ['abstract hip hop', 'abstract hip-hop'],
    'synth rock': ['synth rock', 'synth-rock'],
    'emo pop': ['emo-pop', 'emo pop'],
    'electro pop': ['electro-pop', 'electro pop'],
    'alt pop': ['alt-pop', 'alt pop'],
    'jazz hip hop': ['jazz rap', 'jazz hip-hop'],
    'glitch hop': ['glitch hop', 'glitch-hop'],
    'folk pop': ['folk pop', 'folk-pop'],
    'psychedelic hip hop': ['psychedelic hip hop', 'psychedelic hip-hop'],
    'industrial hip hop': ['industrial hip hop', 'industrial hip-hop'],
    'bro country': ['bro-country', 'bro country'],
    'country rap': ['country rap', 'country-rap'],
    'nu disco': ['nu-disco', 'nu disco'],
    'emo rap': ['emo rap', 'emo-rap', 'emo hip hop'],
    'dark pop': ['dark pop', 'dark-pop'],
    'christian hip hop': ['christian rap', 'christian hip hop', 'christian hip-hop'],
    'chill out': ['chill-out', 'chill out'],
    'instrumental hip hop': ['instrumental hip-hop', 'instrumental hip hop'],
    'nu jazz': ['nu jazz', 'nu-jazz'],
    'lo fi hip hop': ['lo-fi hip-hop', 'lo-fi hip hop'],
    'trap pop': ['trap-pop', 'trap pop'],
    'bit pop': ['bit pop', 'bit-pop'],
    'hip hop soul': ['hip-hop soul', 'soul-rap'],
    'korean hip hop': ['korean hip-hop', 'korean hip hop'],
    'k hip hop': ['k-hip hop', 'k-hiphop', 'k-rap'],
    'electro swing': ['electro swing', 'electro-swing'],
    'pop reggae': ['pop reggae', 'reggae-pop', 'reggae pop'],
    'uk hip hop': ['uk hip-hop', 'uk hip hop'],
    'disco house': ['disco house', 'disco-house'],
    'alt rnb': ['alt-rnb', 'alt rnb'],
    'symphonic hip hop': ['symphonic rap', 'symphonic hip hop'],
    'folk dance': ['folk-dance', 'folk dance'],
    '90s dance': ['dance 90s', '90s dance'],
    'latin hip hop': ['rap latino', 'hip hop latino']
}

# ---------- BUILD LOOKUP ----------
variant_to_canonical = {}
for canonical, variants in GENRE_EQUIVALENTS.items():
    for v in variants:
        variant_to_canonical[v.lower()] = canonical

# ---------- SPACING ----------
def normalize_spacing(g):
    return {
        'poprock': 'pop rock',
        'raprock': 'rap rock',
        'khiphop': 'k hip hop',
        'hiphop': 'hip hop'
    }.get(g, g)

# ---------- FILTER ----------
def is_valid_genre(g):
    if re.search(r'\d', g):
        return False
    if len(g.split()) > 4:
        return False
    return True

# ---------- CLEAN ----------
def clean_genres(g):
    if pd.isna(g):
        return ""

    result = set()

    for x in g.split(','):
        x = x.strip().lower().replace('-', ' ')
        x = normalize_spacing(x)

        if x in variant_to_canonical:
            x = variant_to_canonical[x]

        if is_valid_genre(x):
            result.add(x)

    return ', '.join(sorted(result))

df['genres_cleaned_v2'] = df['genres'].apply(clean_genres)

# ---------- REMOVE ARTIST ----------
def normalize(text):
    return re.sub(r'[^\w\s]', '', str(text).lower())

def is_artist_tag(tag, artist):
    tag_n = normalize(tag)
    artist_n = normalize(artist)

    tag_tokens = set(tag_n.split())
    artist_tokens = set(artist_n.split())

    if tag_n == artist_n:
        return True

    overlap = len(tag_tokens & artist_tokens)
    if overlap >= max(1, len(artist_tokens) - 1):
        return True

    return False

def remove_artist(row):
    if pd.isna(row['genres_cleaned_v2']):
        return ""

    artist = row['main_artist']
    result = []

    for g in row['genres_cleaned_v2'].split(','):
        g = g.strip()
        if not is_artist_tag(g, artist):
            result.append(g)

    return ', '.join(result)

df['genres_final'] = df.apply(remove_artist, axis=1)

# ---------- SAVE ----------
df.to_csv("songs_cleaned_final.csv", index=False)

print("Done. Final cleaned file saved.")

Done. Final cleaned file saved.


## Genre Frequency Analysis

This section analyzes the distribution of genres across the dataset by counting how frequently each genre appears.

---

### Data Loading

- Loads the cleaned dataset containing finalized genre labels  
- Focuses on the column with processed genre information  

---

### Genre Expansion

- Splits comma-separated genre strings into individual entries  
- Flattens the data so each genre is treated as a separate record  
- Removes extra whitespace for consistency  

---

### Frequency Calculation

- Counts how many times each genre appears in the dataset  
- Provides a measure of genre popularity and distribution  

---

### Sorting

- Sorts genres in ascending order of frequency  
- Highlights less frequent (rare) genres at the top  

---

### Tabular Conversion

- Converts the frequency counts into a structured table  
- Uses clear column labels:
  - Genre  
  - Count  

---

### Output

- Displays the genre frequency table  
- Optionally saves the results to a CSV file for further analysis  

---

### Purpose

- Understands genre distribution in the dataset  
- Identifies dominant and rare genres  
- Supports decisions for filtering, weighting, or modeling in recommendation systems  

In [50]:
import pandas as pd

# ---------- LOAD ----------
df = pd.read_csv("songs_cleaned_final.csv")

# ---------- SPLIT + FLATTEN ----------
all_genres = (
    df['genres_final']
    .dropna()
    .str.split(',')
    .explode()           # turns lists into rows
    .str.strip()
)

# ---------- COUNT ----------
genre_counts = all_genres.value_counts()

# ---------- SORT (LOW → HIGH) ----------
genre_counts_sorted = genre_counts.sort_values(ascending=True)

# ---------- CONVERT TO TABLE ----------
genre_table = genre_counts_sorted.reset_index()
genre_table.columns = ['genre', 'count']

# ---------- PRINT ----------
print(genre_table)

# ---------- OPTIONAL: SAVE ----------
genre_table.to_csv("genre_frequency_table.csv", index=False)

                   genre  count
0     free improvisation      1
1               doomstep      1
2         secret setlist      1
3            guadalajara      1
4                   dean      1
...                  ...    ...
4212    alternative rock   3145
4213               indie   3426
4214                rock   3763
4215                 pop   5989
4216             hip hop   6825

[4217 rows x 2 columns]


## Manual Noise Tag Removal

This section removes unwanted or noisy genre tags using a manually curated list.

---

### Background

- A frequency analysis of genres was performed to identify noisy or irrelevant tags  
- These included artist names and other non-genre labels  
- The identified noise tags were stored in a separate file  

---

### Noise Tag Reference

- Loads a list of tags to be removed from an external file  
- Normalizes tags to ensure consistent matching  

---

### Filtering Process

- Iterates through each genre string  
- Splits the string into individual tags  
- Converts tags to a normalized format  
- Removes any tags that appear in the predefined noise list  

---

### Result Generation

- Creates a new column with filtered genre tags  
- Preserves only meaningful and relevant genre information  

---

### Verification

- Displays a sample comparison between original and filtered genres  
- Helps validate the effectiveness of the filtering process  

---

### Final Output

- Saves the cleaned dataset with filtered genres to a new file  

---

### Purpose

- Eliminates residual noise not handled by automated cleaning  
- Improves overall data quality  
- Enhances the reliability of downstream tasks such as clustering and recommendation  

In [56]:
import pandas as pd

# ---------- LOAD ----------
df = pd.read_csv("songs_cleaned_final.csv")
tags_df = pd.read_csv("tags.csv")

# ---------- BUILD REMOVAL SET ----------
tags_to_remove = set(
    tags_df['tags']
    .dropna()
    .str.lower()
    .str.strip()
)

# ---------- CLEAN FUNCTION ----------
def remove_unwanted_tags(genre_string):
    if pd.isna(genre_string):
        return ""

    cleaned = []

    for g in genre_string.split(','):
        g = g.strip().lower()

        if g not in tags_to_remove:
            cleaned.append(g)

    return ', '.join(cleaned)

# ---------- APPLY ----------
df['genres_filtered'] = df['genres_final'].apply(remove_unwanted_tags)

# ---------- OPTIONAL CHECK ----------
print("Sample:\n")
print(df[['genres_final', 'genres_filtered']].head(10))

# ---------- SAVE ----------
df.to_csv("songs_final_filtered.csv", index=False)

print("\nDone. Filtered file saved as songs_final_filtered.csv")

Sample:

                                        genres_final  \
0                            hip hop, pop, r&b, soul   
1  alternative rock, pop, pop punk, punk, punk ro...   
2                                       jazz, oldies   
3  gangsta rap, hip hop, snoop dogg, west coast, ...   
4  alternative rock, california, funk, funk rock,...   
5  alternative rock, happy, pop, pop rock, rock, ...   
6                     dance, hip hop, pop, r&b, soul   
7  alternative rock, pop punk, pop rock, punk, pu...   
8           boybands, love, nostalgia, pop, teen pop   
9  eminem, gangsta rap, hip hop, west coast, west...   

                                     genres_filtered  
0                            hip hop, pop, r&b, soul  
1  alternative rock, pop, pop punk, punk, punk ro...  
2                                       jazz, oldies  
3  gangsta rap, hip hop, west coast, west coast h...  
4  alternative rock, california, funk, funk rock,...  
5  alternative rock, happy, pop, pop rock, r

In [63]:
import pandas as pd
import numpy as np 
import os

df = pd.read_csv("songs_final_filtered.csv")

all_genres = []

for i in df['genres_filtered']:
    if pd.isna(i):  
        continue

    this_genre = i.split(',')
    for j in this_genre:
        j = j.strip()
        if j not in all_genres:
            all_genres.append(j)

genres_df=pd.DataFrame(all_genres)
genres_df.info()
print(all_genres)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       607 non-null    object
dtypes: object(1)
memory usage: 4.9+ KB
['hip hop', 'pop', 'r&b', 'soul', 'alternative rock', 'pop punk', 'punk', 'punk rock', 'rock', 'jazz', 'oldies', 'gangsta rap', 'west coast', 'west coast hip hop', 'california', 'funk', 'funk rock', 'mellow', 'happy', 'pop rock', 'soundtrack', 'dance', 'boybands', 'love', 'nostalgia', 'teen pop', 'acoustic', 'chill', 'indie', 'classic rock', 'g funk', 'classic', 'epic', 'angry', 'metal', 'nu metal', 'rap metal', 'rapcore', 'canadian', 'sexy', 'grunge', 'hard rock', 'post grunge', 'classic country', 'country', 'traditional country', 'dance pop', 'bluegrass', 'folk', 'alternative metal', 'latin', 'party', 'disco', 'funky', 'comedy', 'political', 'sad', 'melancholic', 'modern country', 'rap rock', 'cumbia', 'mexico', 'ska', 'guitar', 'latin 